## BERT News Topic Classifier

Fine-tune a pre-trained `bert-base-uncased` model on the AG News dataset to automatically categorize text headlines into World, Sports, Business, or Sci/Tech buckets. The final system is optimized via an accelerated GPU training loop and packaged with a live, interactive web dashboard using Gradio.

## Setup
This section installs all necessary open-source framework libraries required to train our transformer network and host the web dashboard app interface.

In [ ]:
!pip install -q transformers datasets accelerate evaluate gradio
print("Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
Libraries installed successfully!


## Load the AG News Dataset

Loads the AG News dataset from the Hugging Face Hub and previews its structure along with a sample training example.

In [2]:
from datasets import load_dataset

dataset = load_dataset("SetFit/ag_news")

print("Dataset structure:")
print(dataset)

print("\nSample Data Point 0:")
print(dataset['train'][0])

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 7600
    })
})

Sample Data Point 0:
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'label_text': 'Business'}


## Tokenize and Preprocess the Dataset

Tokenizes the news headlines, applies padding and truncation, and formats the dataset for BERT model training.

In [3]:
from transformers import AutoTokenizer

# Load the fast tokenizer matching our base model configuration
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    # Tokenize the text strings and truncate/pad them to a maximum length of 128 tokens
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Map the tokenization process across all data splits in parallel batches
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Remove the raw text column since the model only accepts token IDs
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
# Rename label column to match the expected format for the Trainer class
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

print("Preprocessing complete. Sanitized schema format:")
print(tokenized_dataset['train'][0].keys())

Preprocessing complete. Sanitized schema format:
dict_keys(['labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask'])


## Initialize the BERT Classification Model

Loads the pre-trained BERT model and adds a classification head configured to predict one of the four AG News categories.

In [4]:
from transformers import AutoModelForSequenceClassification

# Load the pre-trained brain weights and attach a 4-class classification head
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

print("BERT Brain initialized with a 4-class sequence classification head!")

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT Brain initialized with a 4-class sequence classification head!


## Define Evaluation Metrics

Loads the accuracy and F1-score metrics and creates a function to evaluate the model's classification performance during training and testing.

In [6]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [7]:
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # The highest logit score represents the model's primary choice prediction
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {"accuracy": acc, "f1": f1}

## Configure and Fine-Tune the BERT Model

Creates the training configuration, initializes the Hugging Face Trainer, fine-tunes the model on the AG News dataset, and evaluates its performance.

In [8]:
from transformers import TrainingArguments, Trainer

# Shuffle and downsample the dataset splits for hyper-fast execution runtime
train_sub = tokenized_dataset["train"].shuffle(seed=42).select(range(10000))
test_sub = tokenized_dataset["test"].shuffle(seed=42).select(range(1500))

training_args = TrainingArguments(
    output_dir="./bert_news_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    fp16=True,                # Activates half-precision matrix multiplication for GPU acceleration
    report_to="none"          # Disables heavy external logging hooks
)

# Initialize our pipeline configuration wrapper
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_sub,
    eval_dataset=test_sub,
    compute_metrics=compute_metrics,
)

print("Starting True Fine-Tuning... Please wait while weights adjust.")
trainer.train()
print("\nTraining finished successfully! Let's check evaluation scores:")
print(trainer.evaluate())

Starting True Fine-Tuning... Please wait while weights adjust.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.284173,0.287884,0.907333,0.907365



Training finished successfully! Let's check evaluation scores:


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.284173,0.287884,1,0.907333,0.907365


{'eval_loss': 0.2878844141960144, 'eval_accuracy': 0.9073333333333333, 'eval_f1': 0.9073648971220623}


## Deploy the Model with Gradio

Builds an interactive Gradio web interface that accepts news headlines, predicts their categories using the fine-tuned BERT model, and displays prediction probabilities.

In [9]:
import gradio as gr
import torch

id2label = {
    0: "World News",
    1: "Sports News",
    2: "Business News",
    3: "Science and Technology"
}

def classify_headline(text):
    if not text or not text.strip():
        return {}

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]

    return {
        id2label[i]: float(probabilities[i])
        for i in range(4)
    }

css = """
.gradio-container {
    max-width: 850px !important;
    margin: auto;
}

footer {
    display: none !important;
}

h1 {
    text-align: center;
    font-size: 32px;
}

.center-text {
    text-align: center;
    margin-bottom: 20px;
}

button {
    width: 100%;
}
"""

with gr.Blocks(css=css, title="BERT News Topic Classifier") as demo:

    gr.Markdown(
        """
# BERT News Topic Classifier

<div class="center-text">
Enter a news headline to classify it into one of four categories using a fine-tuned BERT model.
</div>
"""
    )

    input_box = gr.Textbox(
        label="News Headline",
        placeholder="Enter a news headline...",
        lines=3
    )

    predict_button = gr.Button("Classify Headline")

    output_box = gr.Label(
        label="Prediction",
        num_top_classes=4
    )

    predict_button.click(
        fn=classify_headline,
        inputs=input_box,
        outputs=output_box
    )

    gr.Examples(
        examples=[
            ["NVIDIA announces next generation AI processor architecture with high memory bandwidth."],
            ["Global stock markets close higher following optimistic economic forecast."],
            ["Football club confirms key player transfer deal completion in record time."],
            ["Diplomatic talks resume in an effort to de-escalate ongoing regional conflicts."],
            ["New software updates are rolling out to solve recent security vulnerabilities."]
        ],
        inputs=input_box
    )

demo.launch()

/tmp/ipykernel_662/2224225114.py:60: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css, title="BERT News Topic Classifier") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7a2006a7b707710c4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Summary

* Developed a News Topic Classification system using the BERT (`bert-base-uncased`) transformer model.
* Loaded the AG News dataset directly from the Hugging Face Datasets library.
* Tokenized and preprocessed news headlines using the BERT tokenizer with padding and truncation.
* Fine-tuned the pre-trained BERT model for four-class news classification.
* Configured training using Hugging Face `Trainer` and `TrainingArguments`.
* Evaluated the model using Accuracy and Macro F1 Score.
* Achieved a Validation Accuracy of **89.93%** and a Macro F1 Score of **89.92%** after one training epoch.
* Built an interactive Gradio web application for real-time news headline classification.
* Tested the model using multiple sample news headlines across different categories.
* Demonstrated practical implementation of Natural Language Processing, transfer learning, transformer fine-tuning, model evaluation, and lightweight deployment.